# HW2 Encoder v3 reproduction
Generate the frozen v3 second view for 767 FAR-Trans assets on a Tesla T4 in FP16. Score the three Encoder variants and four starter baselines locally on the same 100 HW2 users. The Colab inputs contain only as-of asset fields and asset IDs. Customer data and evaluation labels stay local.

This notebook is a runbook. Saved generation results and the resumable cache live in Google Drive. Run the local commands from the repository root with Python 3.11 and Git available.

## 1. Verify the saved starter reference locally

Put the original run's `protocol.json`, `predictions.csv`, and `metrics.json` together in `outputs/hw2-starter-check/`. Use the existing HW2 starter output or obtain an unchanged copy of these three files from the teammate or archive holding that run. The verification command checks the SHA-256 of all three files and recomputes the protocol digest.

```powershell
python -B far-trans-llm/hw2_replay.py verify-reference --starter-reference outputs/hw2-starter-check
```

The expected protocol hash is `50efd4aeb415ba6f0c66dd0793a7c2033cdc32a42b86b58eebbca3e81eaf3417`. The three file hashes are recorded in `REFERENCE_HASHES` in the replay script.

The original starter command is shown below with a fresh, ignored output path. Set the dataset path to the extracted local FAR-Trans folder containing `asset_information.csv`, `customer_information.csv`, `close_prices.csv`, and `transactions.csv`.

```powershell
$farTrans = 'C:/Users/User/Project/data/FAR-Trans'
python -B starter/starter.py run --data-dir $farTrans --output-dir outputs/hw2-starter-fresh --cutoff 2021-06-01 --horizon-days 30 --max-users 100 --cohort warm --candidate-count 50
```

A fresh run checks the protocol, but identical protocol hashes alone cannot recover the original CBF order. The starter uses set iteration to accumulate floating-point scores. The completed replay observed 30 CBF rank positions where the reconstructed scores differed by less than `1e-12`, while all three other baselines reconstructed exactly. Use the archived rankings for exact replay. The script compares fresh CBF with the saved CBF, logs each near-tie discrepancy for that execution, and uses the saved ordering. The number of fresh near-ties can vary across processes.

## 2. Prepare the asset-only bundle

Keep the exact `asset_inputs.json` that accompanied any existing generation export. For a new generation run, create a fresh ignored directory with the command below. The resulting filename is `asset_inputs.json`.

```powershell
$farTrans = 'C:/Users/User/Project/data/FAR-Trans'
python -B far-trans-llm/hw2_replay.py prepare --data-dir $farTrans --starter-reference outputs/hw2-starter-check --output-dir outputs/hw2-encoder-v3-new-prep
```

Open this notebook in Colab and select a Tesla T4 GPU runtime. Upload `asset_inputs.json` from the preparation directory and the four repository files `encoder.py`, `data_adapter.py`, `runtime.py`, and `requirements.txt` from `far-trans-llm/`. Keep the raw dataset local.

In [ ]:
%cd /content
import hashlib
import json
from pathlib import Path

from google.colab import drive, files

uploaded = files.upload()
input_file = Path("asset_inputs.json")
input_hash = hashlib.sha256(input_file.read_bytes()).hexdigest()
spec = json.loads(input_file.read_text(encoding="utf-8"))
assert spec["protocol_hash"] == "50efd4aeb415ba6f0c66dd0793a7c2033cdc32a42b86b58eebbca3e81eaf3417"
assert spec["model"] == "Qwen/Qwen3-1.7B"
assert spec["revision"] == "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
for name in ("encoder.py", "data_adapter.py", "runtime.py", "requirements.txt"):
    expected = spec["source_sha256"][f"far-trans-llm/{name}"]
    source = Path(name).read_bytes().replace(b"\r\n", b"\n")
    assert hashlib.sha256(source).hexdigest() == expected, name

assets = spec["assets"]
assert len(assets) == len({row["asset_id"] for row in assets}) == 767
allowed_fields = {"assetName", "assetCategory", "assetSubCategory", "sector", "industry", "marketID"}
assert all(row["as_of_date"][:10] < spec["cutoff"] and set(row["fields"]) == allowed_fields
           for row in assets)
print("Asset input SHA-256:", input_hash)
drive.mount("/content/drive")

## 3. Load the pinned model in FP16

In a fresh session, run the next cell if the pinned inference packages are missing. If Colab requests a restart, restart and rerun the upload cell. The model cell can reuse an existing `backend` if it passes the same GPU, revision, and dtype checks. Setup metrics stay null in that case.

In [ ]:
%pip install -r /content/requirements.txt

In [ ]:
import importlib
import time
from unittest.mock import patch

import torch
import transformers

import encoder
from runtime import LocalGenerator

encoder = importlib.reload(encoder)
assert torch.__version__.split("+", 1)[0] == "2.11.0", torch.__version__
assert transformers.__version__ == "4.57.6", transformers.__version__
assert torch.cuda.is_available(), "Use a Colab GPU runtime."
gpu = torch.cuda.get_device_name(0)
assert "T4" in gpu, f"Expected a Tesla T4, got {gpu}"

backend = globals().get("backend")
reused_model = backend is not None
setup_seconds = setup_peak_mib = None
if not reused_model:
    original_load = transformers.AutoModelForCausalLM.from_pretrained

    def load_fp16(*args, **kwargs):
        kwargs["torch_dtype"] = torch.float16
        return original_load(*args, **kwargs)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    with patch.object(transformers.AutoModelForCausalLM, "from_pretrained", side_effect=load_fp16):
        backend = LocalGenerator(
            spec["model"], revision=spec["revision"], device="cuda", allow_download=True
        )
    torch.cuda.synchronize()
    setup_seconds = time.perf_counter() - started
    setup_peak_mib = torch.cuda.max_memory_allocated() / 2**20

parameter = next(backend.model.parameters())
assert parameter.is_cuda and parameter.dtype == torch.float16
assert backend.metadata["model"] == spec["model"]
assert backend.metadata["requested_revision"] == backend.metadata["resolved_revision"] == spec["revision"]
assert backend.metadata["do_sample"] is False and backend.metadata["thinking"] is False
runtime = {**backend.metadata, "dtype": str(parameter.dtype), "gpu": gpu}
print(json.dumps(runtime, indent=2))

## 4. Generate or resume from Drive

Use one active Colab session per cache. Each completed response is saved before the next request. To resume after interruption, rerun with the same input file and runtime configuration. Cache keys include the exact prompt and generation limit. The report separates new inference from cache hits and counts shared fallbacks by reason.

In [ ]:
def digest(value):
    data = json.dumps(value, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def write_atomic(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


run_dir = Path("/content/drive/MyDrive/hw2-encoder-v3") / input_hash / digest(runtime)
cache_dir = run_dir / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)


class CachedGenerator:
    def __init__(self):
        self.trace = []

    def generate(self, prompt, max_new_tokens=160):
        key = digest({"prompt": prompt, "max_new_tokens": max_new_tokens})
        path = cache_dir / f"{key}.json"
        cache_hit = path.exists()
        if cache_hit:
            response = json.loads(path.read_text(encoding="utf-8"))
        else:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            response = backend.generate(prompt, max_new_tokens=max_new_tokens)
            torch.cuda.synchronize()
            response["peak_gpu_allocated_mib"] = torch.cuda.max_memory_allocated() / 2**20
            write_atomic(path, response)
        self.trace.append({
            "cache_key": key, "cache_hit": cache_hit, "prompt": prompt, "response": response,
        })
        return response

In [ ]:
from collections import Counter

documents = {row["asset_id"]: encoder.source_text(row["fields"]) for row in assets}
generator = CachedGenerator()
started = time.perf_counter()
qwen, templates, failures, audit = encoder.generate_views(documents, generator, on_invalid="fallback")
wall_seconds = time.perf_counter() - started

called_ids = [i for i in sorted(audit) if audit[i]["actual_word_count"] is not None]
responses = dict(zip(called_ids, generator.trace, strict=True))
assert set(qwen) == set(templates) == set(audit) == set(documents)
assert all(qwen[f["asset_id"]] == templates[f["asset_id"]] == documents[f["asset_id"]]
           for f in failures)
unique = {row["cache_key"]: row["response"] for row in generator.trace}
new_calls = [row for row in generator.trace if not row["cache_hit"]]
summary = {
    "assets": len(documents),
    "accepted_pairs": sum(row["accepted_pair"] for row in audit.values()),
    "fallbacks_by_reason": dict(Counter(row["reason"] for row in failures)),
    "exact_text_duplicate_asset_ids": [i for i, row in audit.items() if row["exact_text_duplicate"]],
    "identical_tfidf_token_count_asset_ids": [
        i for i, row in audit.items() if row["identical_tfidf_token_counts"]
    ],
    "cache_hits_this_session": len(generator.trace) - len(new_calls),
    "real_calls_this_session": len(new_calls),
    "generation_seconds_this_session": sum(row["response"]["seconds"] for row in new_calls),
    "generation_wall_seconds_this_session_including_cache_and_validation": wall_seconds,
    "unique_generation_records_used": len(unique),
    "generation_seconds_for_unique_records_used": sum(r["seconds"] for r in unique.values()),
    "peak_gpu_allocated_mib_during_generation": max(
        (r["peak_gpu_allocated_mib"] for r in unique.values()), default=None
    ),
}
output = run_dir / "encoder_v3_generations.json"
write_atomic(output, {
    "protocol_hash": spec["protocol_hash"],
    "input_sha256": input_hash,
    "runtime": runtime,
    "reused_model": reused_model,
    "model_setup_seconds_including_download": setup_seconds,
    "model_setup_peak_allocated_mib": setup_peak_mib,
    "summary": summary,
    "responses": responses,
    "audit": audit,
    "failures": failures,
})
print(json.dumps(summary, indent=2))
print(output)

In [ ]:
files.download(str(output))

## 5. Replay and score locally

Place the export next to the corresponding input bundle in ignored storage. Run the following for the completed experiment. For a new experiment, substitute its matching bundle and export paths and choose a fresh ignored scoring directory.

```powershell
$farTrans = 'C:/Users/User/Project/data/FAR-Trans'
python -B far-trans-llm/hw2_replay.py score --data-dir $farTrans --starter-reference outputs/hw2-starter-check --asset-inputs outputs/hw2-encoder-v3-prep/asset_inputs.json --generation-export outputs/hw2-encoder-v3-prep/encoder_v3_generations.json --output-dir outputs/hw2-encoder-v3-reproduced
```

The scoring script uses saved responses and the standard library. It verifies the reference hashes, raw data hashes, as-of input fields, frozen sources, prompts, and exact baseline rankings. Source hash checks allow Git's LF/CRLF conversion by hashing in the frozen line endings (LF for Encoder files, CRLF for the starter). The saved starter artifacts still require byte-for-byte matches. The ignored results include `metrics.json`, `predictions.csv`, `per_user.json`, `run_manifest.json`, and `replay_checks.json`. The checks file records Qwen versus template ranking differences and the paired percentile bootstrap settings and intervals. Keep the raw dataset, reference artifacts, generation export, caches, and outputs out of Git.